# Fitting coefficients of the simplified equation of state and comparing with polyTEOS10-bsq the accuracy to TEOS-10

In NEMO, a simplified equation of state is already implemented:

$d_a=\frac{\rho'-\rho_0}{\rho_0}=\frac{1}{\rho_0}[-a_0(1+0.5\lambda_1 T_a + \mu_1 z)T_a + b_0(1-0.5\lambda_2 S_a -\mu_2 z)S_a -\nu T_a S_a]$

The coefficients of this equation have not been fitted in our ranges of interest, i.e., 0 to 1 g/kg for salinity, -1 to 10°C for temperature and 0 to 750 db for pressure. In this notebook we fit these coefficients according to our ranges of interest, considering the TEOS10 formulation as reference (gsw.density.rho_t_exact). 

gsw.density.rho_t_exact computes the in-situ density, while da is an anomaly of density. So if we rewrite the first equation to express the in-situ density we have:

$\rho=\rho_0(z)+\rho'$

with

$\rho'=\rho_0 - a_0(1+0.5\lambda_1 T_a + \mu_1 z)T_a + b_0(1-0.5\lambda_2 S_a -\mu_2 z)S_a -\nu T_a S_a$

and $\rho_0(z)$ a vertical profile of reference:

$\rho_0(z)=\rho(S_0,T_0,z)-\rho(S_0,T_0,0)=\rho(S_0,T_0,z)-\rho_0$

In [1]:
#import the libraries needed
import gsw
import numpy as np
from math import sqrt
from scipy.optimize import curve_fit

### Define the salinity, temperature and density reference values and the ranges of interest for the fit

In [2]:
#define the salinity and temperature reference values
S0 = 0.5 ; T0 = 4 #Absolute Salinity in g/kg and Conservative Temperature in °C
t0 = gsw.conversions.t_from_CT(S0,T0,0) #convert Conservative Temperature to in-situ temperature
print(gsw.density.rho_t_exact(S0,t0,0))

1000.3763825587874


In [3]:
#define the in-situ density of reference
rho0=1000.38

In [4]:
#define our ranges of interest
SA_min = 0 ; SA_max = 1 #g/kg
CT_min = -1 ; CT_max = 10 #°C
p_min = 0 ; p_max = 750 #dbar

print("salinity :",SA_min,SA_max)
print("temperature :",CT_min,CT_max)
print("pressure :",p_min,p_max)

salinity : 0 1
temperature : -1 10
pressure : 0 750


### Fitting the coefficients of da

In [5]:
#define the function of the fit
def rho_prime(X,a0,b0,l1,l2,nu,mu1,mu2):
    """
    Calculate rho prime using the simplified equation of state presented in Fiol et al. "Ice shelf basal melting in giant proglacial lakes".

    Parameters
    ----------
    X : contains the salinity, temperature and depth values
        X[0,:] = Absolute Salinity (g/kg)
        X[1,:] = Conservative Temperature (°C)
        X[2,:] = depth (m) (equal to zero at the surface and positive towards depth)
    a0, b0, l1, l2, nu, mu1, mu2 : some coefficients

    Returns
    -------
    rho : rho prime (kg/m3)
    """
    
    global rho0, S0, T0 #density, salinity and temperature of reference
    
    S_anom=X[0,:]-S0 #anomaly of salinity
    T_anom=X[1,:]-T0 #anomaly of temperature

    #compute rho prime
    rho=rho0 - a0*(1+0.5*l1*T_anom+mu1*X[2,:])*T_anom + b0*(1-0.5*l2*S_anom-mu2*X[2,:])*S_anom - nu*T_anom*S_anom
    return rho

In [6]:
#define the number of values for the fit
taille=550

In [7]:
#create xdata
p=np.repeat(np.linspace(p_min,p_max,taille),taille**2)
xdata=np.ones((3,taille**3))*np.nan
xdata[0,:]=np.tile(np.linspace(SA_min,SA_max,taille),taille**2)
xdata[1,:]=np.tile(np.repeat(np.linspace(CT_min,CT_max,taille),taille),taille)
xdata[2,:]=p/(rho0*9.80665*10**(-4)) #convert pressure to depth using the hydrostatic equilibrium 

In [8]:
#create ydata
xt_insitu=gsw.conversions.t_from_CT(xdata[0,:],xdata[1,:],p)
profil_vertical=gsw.density.rho_t_exact(S0,t0,p) -rho0
ydata=gsw.density.rho_t_exact(xdata[0,:],xt_insitu,p) - profil_vertical

In [9]:
#fitting
fit_rho_prime=curve_fit(rho_prime, xdata, ydata, full_output=True)

In [10]:
#display the fitted parameters 
print(fit_rho_prime[0])

[-2.27362059e-03  8.02280183e-01 -6.12138918e+00  1.37255690e-02
  3.31755664e-03 -1.27918389e-02  1.13493455e-05]


In [11]:
#evaluate the fit
Y_rho_prime=rho_prime(xdata,-2.274e-03,8.0228e-01,-6.121,1.3726e-02,3.3176e-03,-1.279e-02,1.135e-05)
RMSE_rho_prime=sqrt(np.mean((Y_rho_prime-ydata)**2))
print(RMSE_rho_prime)

0.0021174915666067676


In [12]:
#compute density anomalies
anomaly_rho_prime=Y_rho_prime/rho0 -1
anomaly_ydata=ydata/rho0 -1

In [13]:
#evaluate the fit in density anomalies
RMSE_anomaly_rho_prime=sqrt(np.mean((anomaly_rho_prime-anomaly_ydata)**2))
print(RMSE_anomaly_rho_prime)

2.1166872254610804e-06


### Comparison of polyTEOS10-bsq with TEOS10

In [14]:
#define polyTEOS10-bsq
def polyTEOS10_bsq_vertical_ref_profil_opti(Z):
    """
    Calculate vertical reference profile of polyTEOS10_bsq (Roquet et al., 2015)

    Parameters
    ----------
    Z : depth (m) (equal to zero at the surface and negative towards depth)

    Returns
    -------
    r0 : vertical reference profile of density (kg/m3)
    """
    Zu= 1e4;zz= -np.divide(Z,Zu)
    R00= 4.6494977072e+01;R01= -5.2099962525e+00;R02= 2.2601900708e-01;
    R03= 6.4326772569e-02;R04= 1.5616995503e-02;R05= -1.7243708991e-03;
    r0= np.multiply((np.multiply((np.multiply((np.multiply((np.multiply((np.multiply(R05,zz)+R04),zz)+R03),zz)+R02),zz)+R01),zz)+R00),zz)
    return(r0)

def polyTEOS10_bsq_density_anomaly_opti(SA,CT,Z):
    """
    Calculate rho prime using polyTEOS10_bsq (Roquet et al., 2015)

    Parameters
    ----------
    SA : Absolute Salinity (g/kg)
    CT : Conservative Temperature (°C)
    Z : depth (m) (equal to zero at the surface and negative towards depth)

    Returns
    -------
    r : rho prime (kg/m3)
    """
    SAu= 40*35.16504/35;CTu= 40;Zu= 1e4
    deltaS= 32
    R000= 8.0189615746e+02;R100= 8.6672408165e+02;R200= -1.7864682637e+03;
    R300= 2.0375295546e+03;R400= -1.2849161071e+03;R500= 4.3227585684e+02;
    R600= -6.0579916612e+01;R010= 2.6010145068e+01;R110= -6.5281885265e+01;
    R210= 8.1770425108e+01;R310= -5.6888046321e+01;R410= 1.7681814114e+01;
    R510= -1.9193502195e+00;R020= -3.7074170417e+01;R120= 6.1548258127e+01;
    R220= -6.0362551501e+01;R320= 2.9130021253e+01;R420= -5.4723692739e+00;
    R030= 2.1661789529e+01;R130= -3.3449108469e+01;R230= 1.9717078466e+01;
    R330= -3.1742946532e+00;R040= -8.3627885467e+00;R140= 1.1311538584e+01;
    R240= -5.3563304045e+00;R050= 5.4048723791e-01;R150= 4.8169980163e-01;
    R060= -1.9083568888e-01;R001= 1.9681925209e+01;R101= -4.2549998214e+01;
    R201= 5.0774768218e+01;R301= -3.0938076334e+01;R401= 6.6051753097e+00;
    R011= -1.3336301113e+01;R111= -4.4870114575e+00;R211= 5.0042598061e+00;
    R311= -6.5399043664e-01;R021= 6.7080479603e+00;R121= 3.5063081279e+00;
    R221= -1.8795372996e+00;R031= -2.4649669534e+00;R131= -5.5077101279e-01;
    R041= 5.5927935970e-01;R002= 2.0660924175e+00;R102= -4.9527603989e+00;
    R202= 2.5019633244e+00;R012= 2.0564311499e+00;R112= -2.1311365518e-01;
    R022= -1.2419983026e+00;R003= -2.3342758797e-02;R103= -1.8507636718e-02;
    R013= 3.7969820455e-01;
    !
    ss= np.sqrt(np.divide(np.add(SA,deltaS),SAu));
    tt= np.divide(CT,CTu);
    zz= - np.divide(Z,Zu);
    rz3= np.multiply(R013,tt) + np.multiply(R103,ss)+ R003
    rz2= np.multiply((np.multiply(R022,tt)+np.multiply(R112,ss)+R012),tt)+np.multiply((np.multiply(R202,ss)+R102),ss)+R002
    rz1= np.multiply((np.multiply((np.multiply((np.multiply(R041,tt)+np.multiply(R131,ss)+R031),tt) + np.multiply((np.multiply(R221,ss)+R121),ss)+R021),tt) + np.multiply((np.multiply((np.multiply(R311,ss)+R211),ss)+R111),ss)+R011),tt) + np.multiply((np.multiply((np.multiply((np.multiply(R401,ss)+R301),ss)+R201),ss)+R101),ss)+R001
    rz0= np.multiply((np.multiply((np.multiply((np.multiply((np.multiply((np.multiply(R060,tt)+np.multiply(R150,ss)+R050),tt) + np.multiply((np.multiply(R240,ss)+R140),ss)+R040),tt) + np.multiply((np.multiply((np.multiply(R330,ss)+R230),ss)+R130),ss)+R030),tt) + np.multiply((np.multiply((np.multiply((np.multiply(R420,ss)+R320),ss)+R220),ss)+R120),ss)+R020),tt) + np.multiply((np.multiply((np.multiply((np.multiply((np.multiply(R510,ss)+R410),ss)+R310),ss)+R210),ss)+R110),ss)+R010),tt) +np.multiply((np.multiply((np.multiply((np.multiply((np.multiply((np.multiply(R600,ss)+R500),ss)+R400),ss)+R300),ss)+R200),ss)+R100),ss)+R000
    r= np.multiply((np.multiply((np.multiply(rz3,zz)+ rz2),zz)+ rz1),zz)+ rz0
    return (r)

def polyTEOS10_bsq_density_opti(SA,CT,Z):
    """
    Calculate in situ denisty using polyTEOS10_bsq (Roquet et al., 2015)

    Parameters
    ----------
    SA : Absolute Salinity (g/kg)
    CT : Conservative Temperature (°C)
    Z : depth (m) (equal to zero at the surface and negative towards depth)

    Returns
    -------
    in situ density (kg/m3)
    """
    return (polyTEOS10_bsq_vertical_ref_profil_opti(Z)+polyTEOS10_bsq_density_anomaly_opti(SA,CT,Z))

#Checkvaluefor Z= −1000m:r0= 4.59763035kgm−3
print(polyTEOS10_bsq_vertical_ref_profil_opti([-1000,-1000,-1000]))
#Checkvaluesfor SA= 30g/kg, CT= 10◦C, Z= −1000m:
#r= 1022.85377kgm−3, ρ= 1027.45140kgm−3
print(polyTEOS10_bsq_density_anomaly_opti([30,30,30],[10,10,10],[-1000,-1000,-1000]))
print(polyTEOS10_bsq_density_opti([30,30,30],[10,10,10],[-1000,-1000,-1000]))
#Check value: r = 1028.21993233072 kg/m^3 for Z=-3000m, CT=3°C, SA=35.5 g/kg
print(polyTEOS10_bsq_density_anomaly_opti([35.5,35.5,35.5],[3,3,3],[-3000,-3000,-3000]))


[4.59763035 4.59763035 4.59763035]
[1022.85377082 1022.85377082 1022.85377082]
[1027.45140117 1027.45140117 1027.45140117]
[1028.21993233 1028.21993233 1028.21993233]


**Compute RMSEs**

In [15]:
#create xdata2 (same as xdata but with negative depths)

xdata2=np.copy(xdata)
xdata2[2,:]=-xdata[2,:]

In [16]:
#some checks
print(np.sum(xdata[0,:]-xdata2[0,:]))
print(np.sum(xdata[1,:]-xdata2[1,:]))
print(np.sum(xdata[2,:]+xdata2[2,:]))
print(np.sum(xt_insitu-gsw.conversions.t_from_CT(xdata2[0,:],xdata2[1,:],p)))

0.0
0.0
0.0
0.0


In [17]:
#compare vertical profile
profil_vertical_TEOS10=gsw.density.rho_t_exact(S0,t0,p) -rho0
profil_vertical_polyTEOS10_bsq=polyTEOS10_bsq_vertical_ref_profil_opti(xdata2[2,:])
diff_profil_vertical=profil_vertical_polyTEOS10_bsq-profil_vertical_TEOS10

    #RMSE
RMSE_profil_vertical=sqrt(np.mean(diff_profil_vertical**2))
print(RMSE_profil_vertical)

0.0879102151341272


In [18]:
#compare rho_prime
r_TEOS10=gsw.density.rho_t_exact(xdata[0,:],xt_insitu,p) - profil_vertical_TEOS10
r_polyTEOS10_bsq=polyTEOS10_bsq_density_anomaly_opti(xdata2[0,:],xdata2[1,:],xdata2[2,:])
diff_r=r_polyTEOS10_bsq-r_TEOS10

    #RMSE
RMSE_r=sqrt(np.mean(diff_r**2))
print(RMSE_r)

0.12872686879469902


In [19]:
#compare da
da_TEOS10=r_TEOS10/rho0 -1
da_polyTEOS10_bsq=r_polyTEOS10_bsq/rho0 -1
diff_da=da_polyTEOS10_bsq-da_TEOS10

    #RMSE
RMSE_da=sqrt(np.mean(diff_da**2))
print(RMSE_da)

0.000128677971165656


In [21]:
#compare in situ density
density_TEOS10=gsw.density.rho_t_exact(xdata[0,:],xt_insitu,p)
density_polyTEOS10_bsq=polyTEOS10_bsq_density_opti(xdata2[0,:],xdata2[1,:],xdata2[2,:])
diff_density=density_polyTEOS10_bsq-density_TEOS10

    #RMSE
RMSE_density=sqrt(np.mean(diff_density**2))
print(RMSE_density)

0.04082963344505522


In [22]:
#display ranges of da computed with TEOS10
print(np.min(da_TEOS10))
print(np.max(da_TEOS10))

-0.0007503512367207366
0.00041542433746788454
